# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will start with **Logistic Regression** because the question is a ranking question: which content should be reviewed first?

The model will produce a probability score for positive observed movement in April. I can rank content by that probability and evaluate the ranking with **Precision@50**.

Logistic Regression is a useful first learned model because it is relatively simple and interpretable. It also gives a clear comparison against the Week-4 rule baseline before adding a more complex model.

The model will use only March 2026 information available at the decision cutoff. April impressions will be used only to define the observed outcome and evaluate the ranking.

The observed outcome is:

**positive movement = April impressions > March impressions.**

This is an observed future outcome for evaluation, not a feature.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1: Prepare the modeling dataset

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

# Connect DuckDB and authenticate with Hugging Face
con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face")

rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 feature window
march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(COALESCE(ga4_sessions, 0)) AS march_ga4_sessions,
        SUM(COALESCE(ga4_engaged_sessions, 0)) AS march_ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
""").df()

# April 2026 future outcome window
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
""").df()

# Content metadata
content = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
""").df()

# Join March features, content metadata, and April outcome
df = (
    march
    .merge(
        content,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

# Content age at March 31, 2026
df["content_age_days"] = (
    pd.Timestamp("2026-03-31")
    - pd.to_datetime(df["content_created_date"])
).dt.days

# March CTR
df["march_ctr"] = np.where(
    df["march_impressions"] > 0,
    df["march_clicks"] / df["march_impressions"],
    0
)

# Match the Week-4 candidate universe
df = df[df["march_impressions"] > 0].copy()

# Future observed outcome — NOT a feature
df["positive_movement"] = (
    df["april_impressions"] > df["march_impressions"]
).astype(int)

print("Modeling rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print(
    "Positive movement rate:",
    round(df["positive_movement"].mean() * 100, 2),
    "%"
)

print()
print("Feature columns:")
print([
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
    "march_ctr",
    "content_age_days"
])

Token loaded: True
DuckDB connected to Hugging Face


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 176737
Unique clients: 47
Positive movement rate: 34.87 %

Feature columns:
['march_impressions', 'march_clicks', 'march_avg_position', 'march_ga4_sessions', 'march_ga4_engaged_sessions', 'march_ctr', 'content_age_days']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **client-grouped 80/20 holdout split**.

The split is grouped by `client_hash_id`, so content from the same client cannot appear in both training and test data.

This is more honest for this question because the model should be tested on clients it did not train on, rather than getting an advantage from seeing other content from the same client during training.

The March 2026 features are available before the March 31 decision cutoff. The April 2026 positive-movement outcome is used only for evaluation.

The Week-4 baseline is a rule-based ranking, not a trained model, so I will calculate its score on this **same held-out test set** for a fair comparison.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2: Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Features available before the decision cutoff
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
    "march_ctr",
    "content_age_days"
]

X = df[feature_cols].copy()
y = df["positive_movement"].copy()
groups = df["client_hash_id"]

# 80/20 split, keeping each client entirely in one side
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Clients in both sets:", len(train_clients & test_clients))
print()
print("Training positive rate:", round(y_train.mean() * 100, 2), "%")
print("Test positive rate:", round(y_test.mean() * 100, 2), "%")


Training rows: 138309
Test rows: 38428
Training clients: 37
Test clients: 10
Clients in both sets: 0

Training positive rate: 33.35 %
Test positive rate: 40.34 %


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3: Train Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

# Logistic Regression pipeline
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train only on the training clients
logistic_model.fit(X_train, y_train)

# Probability of positive movement on unseen test clients
model_scores = logistic_model.predict_proba(X_test)[:, 1]


def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)
    top_k = order[:k]

    return y_true[top_k].mean()


p20 = precision_at_k(y_test, model_scores, 20)
p50 = precision_at_k(y_test, model_scores, 50)

print("Logistic Regression trained successfully")
print("Precision@20:", round(p20 * 100, 2), "%")
print("Precision@50:", round(p50 * 100, 2), "%")


Logistic Regression trained successfully
Precision@20: 65.0 %
Precision@50: 72.0 %


In [12]:
# Section 3: Week-4 baseline on the same test set

baseline_test = df.iloc[test_idx].copy()

# Reproduce the exact Week-4 baseline scoring rule
baseline_test["baseline_score"] = 0

# March impression signal
baseline_test.loc[
    baseline_test["march_impressions"] < 10,
    "baseline_score"
] += 2

baseline_test.loc[
    (baseline_test["march_impressions"] >= 10) &
    (baseline_test["march_impressions"] < 50),
    "baseline_score"
] += 1

# Content age signal
baseline_test.loc[
    baseline_test["content_age_days"] >= 365,
    "baseline_score"
] += 2

baseline_test.loc[
    (baseline_test["content_age_days"] >= 180) &
    (baseline_test["content_age_days"] < 365),
    "baseline_score"
] += 1


# Calculate baseline Precision@20 and Precision@50
baseline_scores = baseline_test["baseline_score"].to_numpy()
baseline_y = baseline_test["positive_movement"].to_numpy()

baseline_p20 = precision_at_k(baseline_y, baseline_scores, 20)
baseline_p50 = precision_at_k(baseline_y, baseline_scores, 50)

print("Week-4 baseline evaluated on the same test set")
print("Precision@20:", round(baseline_p20 * 100, 2), "%")
print("Precision@50:", round(baseline_p50 * 100, 2), "%")

Week-4 baseline evaluated on the same test set
Precision@20: 50.0 %
Precision@50: 30.0 %


In [13]:
# Final comparison: Week-4 baseline vs Logistic Regression

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        p20
    ],
    "Precision@50": [
        baseline_p50,
        p50
    ]
})

# Show percentages
comparison["Precision@20"] = (
    comparison["Precision@20"] * 100
).round(2)

comparison["Precision@50"] = (
    comparison["Precision@50"] * 100
).round(2)

print("Base rate on test set:", round(y_test.mean() * 100, 2), "%")
print()
print(comparison.to_string(index=False))

Base rate on test set: 40.34 %

             Method  Precision@20  Precision@50
    Week-4 baseline          50.0          30.0
Logistic Regression          65.0          72.0


### Model vs baseline

Logistic Regression outperformed the Week-4 rule baseline on the same held-out test clients.

Precision@20 improved from **50.0%** with the baseline to **65.0%** with Logistic Regression.

Precision@50 improved from **30.0%** with the baseline to **72.0%** with Logistic Regression.

The test-set positive rate was **40.34%**, so both methods were evaluated against the same observed outcome and the same client-grouped test set.

For this dataset, the learned model provides a stronger ranking than the simple Week-4 rule. I will still inspect errors and feature importance before treating the model as a useful decision-support method.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4: Errors and interpretation

# Get Logistic Regression coefficients
model = logistic_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0]
})

feature_importance["Abs_coefficient"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "Abs_coefficient",
    ascending=False
)

print("Top features by absolute Logistic Regression coefficient:")
print(
    feature_importance[
        ["Feature", "Coefficient"]
    ].to_string(index=False)
)

# Build test-set results for error analysis
error_analysis = df.iloc[test_idx].copy()
error_analysis["model_score"] = model_scores
error_analysis["predicted_positive"] = (
    error_analysis["model_score"] >= 0.5
).astype(int)

# False positives: model predicted positive, but April did not increase
false_positives = error_analysis[
    (error_analysis["predicted_positive"] == 1) &
    (error_analysis["positive_movement"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print()
print("Top 3 false positives:")
print(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "april_impressions",
            "content_age_days",
            "model_score"
        ]
    ].head(3).to_string(index=False)
)

# False negatives: model predicted negative, but April did increase
false_negatives = error_analysis[
    (error_analysis["predicted_positive"] == 0) &
    (error_analysis["positive_movement"] == 1)
].sort_values(
    "model_score",
    ascending=False
)

print()
print("Top 3 false negatives:")
print(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "april_impressions",
            "content_age_days",
            "model_score"
        ]
    ].head(3).to_string(index=False)
)

Top features by absolute Logistic Regression coefficient:
                   Feature  Coefficient
              march_clicks     0.512647
          content_age_days    -0.360769
        march_ga4_sessions    -0.224789
         march_impressions    -0.202636
        march_avg_position     0.178666
march_ga4_engaged_sessions    -0.023815
                 march_ctr    -0.015697

Top 3 false positives:
         client_hash_id          content_hash_id  march_impressions  april_impressions  content_age_days  model_score
client_73cda7b4e4f265ea content_512dbad65bd5ade9           154358.0           142635.0               187     1.000000
client_73cda7b4e4f265ea content_85703b835ab9e744           120868.0           115430.0                56     1.000000
client_73cda7b4e4f265ea content_5267d90f451c6edc            51920.0            51001.0                56     0.999976

Top 3 false negatives:
         client_hash_id          content_hash_id  march_impressions  april_impressions  content_age_da

### Error analysis and interpretation

The model relies most on **March clicks**, followed by **content age**, **GA4 sessions**, and **March impressions** based on the absolute Logistic Regression coefficients.

The strongest positive coefficient was `march_clicks` (+0.5126). `content_age_days` had a negative coefficient (-0.3608), while `march_ga4_sessions` was positive (+0.2248) and `march_impressions` was negative (-0.2064).

The model still makes difficult predictions.

One false positive had **154,358 March impressions and 142,635 April impressions**. The model gave it a score of **1.00**, but April impressions actually decreased.

A second false positive had **120,868 March impressions and 115,430 April impressions**, also with a model score of **1.00**.

A third false positive had **51,920 March impressions and 51,001 April impressions**, with a model score of **1.00**. These cases show that a high model score does not guarantee positive movement, especially for high-volume content where small declines can still be classified as negative.

The false negatives also show uncertainty near the decision boundary. For example, one item increased from **6,413 to 11,373 impressions**, but its model score was **0.499962**, just below the 0.5 threshold.

Overall, the model provides a stronger ranking than the Week-4 baseline, but the errors show that the observed April movement is not perfectly predictable from the March features alone. The model should therefore be treated as **decision support**, not a guarantee.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.